# Audio Enhancement Experiments for Video Speech

This Colab notebook runs six exploratory enhancement pipelines on one uploaded video:

1. DeepFilterNet -> Resemble Enhance
2. FFmpeg extract -> Resemble Enhance -> loudnorm -> remux
3. FFmpeg extract -> Resemble Enhance
4. DeepFilterNet or Resemble denoise -> LavaSR -> loudnorm
5. FFmpeg extract -> LavaSR
6. FFmpeg extract -> LavaSR -> loudnorm

Use a GPU runtime when possible: `Runtime > Change runtime type > T4 GPU`.
        


## 1. Install Runtime Dependencies

Run this cell first. The model installs can take a few minutes on a fresh Colab runtime.
        


In [ ]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg wget
%pip install -q --upgrade pip
%pip install -q git+https://github.com/resemble-ai/resemble-enhance.git soundfile librosa pandas
%pip install -q git+https://github.com/ysharma3501/LavaSR.git
            


## 2. Imports, Paths, and Experiment Settings
        


In [ ]:
import os
import sys
import json
import shutil
import subprocess
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from IPython.display import Audio, Video, display, HTML

ROOT = Path("/content/audio_experiments")
INPUT_DIR = ROOT / "input"
WORK_DIR = ROOT / "work"
OUTPUT_DIR = ROOT / "outputs"
LOG_DIR = ROOT / "logs"
BIN_DIR = ROOT / "bin"

for directory in [INPUT_DIR, WORK_DIR, OUTPUT_DIR, LOG_DIR, BIN_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

RESULTS = []
VIDEO_PATH = None
BASELINE_WAV = WORK_DIR / "baseline_48k.wav"

# Change this to "resemble_denoise" if you want the denoise -> LavaSR branch
# to use Resemble's denoise-only mode instead of DeepFilterNet.
DENOISE_FOR_LAVASR = "deepfilternet"  # "deepfilternet" or "resemble_denoise"

# LavaSR's README recommends denoise=False by default and enabling it only when needed.
LAVASR_DENOISE = False
LAVASR_BATCH = True

try:
    import torch
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("CPU runtime detected. The notebook can run, but model steps may be slow.")
except Exception as exc:
    print("Could not inspect torch runtime:", exc)
        


## 3. Upload or Mount the Input Video

Option A uploads a local video into the Colab session. Option B copies a video from Google Drive.
        


In [ ]:
USE_GOOGLE_DRIVE = False
DRIVE_VIDEO_PATH = ""  # Example: "/content/drive/MyDrive/input_video.mp4"

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    source_video = Path(DRIVE_VIDEO_PATH)
    if not source_video.exists():
        raise FileNotFoundError(f"Drive video not found: {source_video}")
    target_video = INPUT_DIR / source_video.name
    shutil.copy2(source_video, target_video)
else:
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No video uploaded.")
    uploaded_name = next(iter(uploaded.keys()))
    uploaded_path = Path(uploaded_name)
    target_video = INPUT_DIR / uploaded_path.name
    if target_video.exists():
        target_video.unlink()
    shutil.move(str(uploaded_path), target_video)

VIDEO_PATH = target_video
print("Using video:", VIDEO_PATH)
print("Video size MB:", round(VIDEO_PATH.stat().st_size / 1024 / 1024, 2))
        


## 4. Shared FFmpeg and Filesystem Helpers
        


In [ ]:
def run_cmd(cmd, log_name=None, check=True):
    cmd = [str(part) for part in cmd]
    print("+", " ".join(cmd))
    proc = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if log_name:
        (LOG_DIR / log_name).write_text(proc.stdout, encoding="utf-8")
    if proc.returncode != 0 and check:
        raise RuntimeError(proc.stdout)
    return proc.stdout


def safe_reset_dir(path):
    path = Path(path)
    resolved = path.resolve()
    root_resolved = ROOT.resolve()
    if resolved != root_resolved and root_resolved not in resolved.parents:
        raise ValueError(f"Refusing to reset directory outside {ROOT}: {path}")
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def existing_wavs(directory):
    directory = Path(directory)
    if not directory.exists():
        return set()
    return {p.resolve() for p in directory.rglob("*.wav")}


def newest_new_wav(directory, before):
    after = existing_wavs(directory)
    new_files = list(after - set(before))
    if not new_files:
        candidates = list(Path(directory).rglob("*.wav"))
        if not candidates:
            raise FileNotFoundError(f"No WAV output found under {directory}")
        new_files = candidates
    return max(new_files, key=lambda p: p.stat().st_mtime)


def extract_audio(video_path, output_wav=BASELINE_WAV):
    run_cmd([
        "ffmpeg", "-y", "-i", video_path,
        "-vn", "-ac", "1", "-ar", "48000", "-sample_fmt", "s16",
        output_wav,
    ], "extract_audio.log")
    return Path(output_wav)


def make_comparison_wav(input_wav, output_wav):
    run_cmd([
        "ffmpeg", "-y", "-i", input_wav,
        "-ac", "1", "-ar", "48000", "-sample_fmt", "s16",
        output_wav,
    ], f"{Path(output_wav).stem}_comparison.log")
    return Path(output_wav)


def loudnorm_wav(input_wav, output_wav, i=-16, tp=-1.5, lra=11):
    run_cmd([
        "ffmpeg", "-y", "-i", input_wav,
        "-af", f"loudnorm=I={i}:TP={tp}:LRA={lra}",
        "-ac", "1", "-ar", "48000", "-sample_fmt", "s16",
        output_wav,
    ], f"{Path(output_wav).stem}_loudnorm.log")
    return Path(output_wav)


def remux_video(video_path, audio_wav, output_mp4):
    run_cmd([
        "ffmpeg", "-y", "-i", video_path, "-i", audio_wav,
        "-map", "0:v:0", "-map", "1:a:0",
        "-c:v", "copy", "-c:a", "aac", "-b:a", "192k",
        "-shortest", output_mp4,
    ], f"{Path(output_mp4).stem}_remux.log")
    return Path(output_mp4)
        


## 5. DeepFilterNet Binary Setup

The Python `deepfilternet` package can fail on Colab when its Rust-backed `deepfilterlib` dependency has no compatible wheel. This notebook uses the upstream Linux `deep-filter` release binary instead, which avoids Python package metadata/build failures.
            


In [ ]:
DEEP_FILTER_VERSION = "0.5.6"
DEEP_FILTER_BINARY_URL = "https://github.com/Rikorose/DeepFilterNet/releases/download/v0.5.6/deep-filter-0.5.6-x86_64-unknown-linux-musl"
DEEP_FILTER_BIN = BIN_DIR / "deep-filter"


def install_deepfilter_binary():
    if DEEP_FILTER_BIN.exists():
        print("DeepFilterNet binary already exists:", DEEP_FILTER_BIN)
    else:
        run_cmd(
            ["wget", "-q", "-O", DEEP_FILTER_BIN, DEEP_FILTER_BINARY_URL],
            "install_deepfilter_binary.log",
        )
        DEEP_FILTER_BIN.chmod(0o755)

    version_output = run_cmd(
        [DEEP_FILTER_BIN, "--version"],
        "deepfilter_binary_version.log",
        check=False,
    )
    if version_output.strip():
        print(version_output)
    return DEEP_FILTER_BIN


install_deepfilter_binary()
            


## 6. Inspect Media and Extract Baseline WAV

The baseline extraction is mono, 48 kHz, 16-bit PCM so every model starts from the same audio.
        


In [ ]:
if VIDEO_PATH is None or not Path(VIDEO_PATH).exists():
    raise RuntimeError("Run the upload/mount cell first.")

probe_output = run_cmd(["ffprobe", "-hide_banner", "-i", VIDEO_PATH], "ffprobe_input.log", check=False)
print(probe_output)

BASELINE_WAV = extract_audio(VIDEO_PATH)
print("Baseline WAV:", BASELINE_WAV)
display(Audio(filename=str(BASELINE_WAV)))
        


## 7. Model Runner Functions

These wrappers keep the rest of the notebook independent from each model's output naming convention.
        


In [ ]:
def run_deepfilternet(input_wav, out_dir):
    if "DEEP_FILTER_BIN" not in globals():
        raise RuntimeError("Run the DeepFilterNet binary setup cell before DeepFilterNet pipelines.")

    if not Path(DEEP_FILTER_BIN).exists():
        install_deepfilter_binary()

    out_dir = safe_reset_dir(out_dir)
    before = existing_wavs(out_dir)
    log_base = f"{out_dir.parent.name}_{out_dir.name}_deepfilternet.log"
    run_cmd([DEEP_FILTER_BIN, "--output-dir", out_dir, input_wav], log_base)
    return newest_new_wav(out_dir, before)

def get_resemble_command():
    import importlib.util

    for executable_name in ["resemble-enhance", "resemble_enhance"]:
        cli_path = shutil.which(executable_name)
        if cli_path:
            print("Using Resemble Enhance CLI:", cli_path)
            return [cli_path]

    if importlib.util.find_spec("resemble_enhance") is not None:
        print("Resemble Enhance CLI was not found on PATH; using python module fallback.")
        return [sys.executable, "-m", "resemble_enhance.enhancer.__main__"]

    raise RuntimeError(
        "Resemble Enhance is not importable. Restart the Colab runtime, run the install cell, "
        "and confirm it installs git+https://github.com/resemble-ai/resemble-enhance.git."
    )


def run_resemble(input_wav, out_dir, denoise_only=False):
    out_dir = safe_reset_dir(out_dir)
    in_dir = out_dir / "input"
    result_dir = out_dir / "result"
    in_dir.mkdir(parents=True, exist_ok=True)
    result_dir.mkdir(parents=True, exist_ok=True)

    copied_input = in_dir / Path(input_wav).name
    shutil.copy2(input_wav, copied_input)

    before = existing_wavs(result_dir)
    cmd = get_resemble_command() + [in_dir, result_dir]
    if denoise_only:
        cmd.append("--denoise_only")
    suffix = "denoise_only" if denoise_only else "enhance"
    run_cmd(cmd, f"{out_dir.parent.name}_{out_dir.name}_resemble_{suffix}.log")
    return newest_new_wav(result_dir, before)


LAVA_MODEL = None


def get_lava_model():
    global LAVA_MODEL
    if LAVA_MODEL is not None:
        return LAVA_MODEL
    import torch
    from LavaSR.model import LavaEnhance2

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Loading LavaSR on", device)
    LAVA_MODEL = LavaEnhance2("YatharthS/LavaSR", device)
    return LAVA_MODEL


def run_lavasr(input_wav, output_wav, denoise=LAVASR_DENOISE, batch=LAVASR_BATCH):
    import soundfile as sf

    output_wav = Path(output_wav)
    output_wav.parent.mkdir(parents=True, exist_ok=True)

    lava_model = get_lava_model()
    input_audio, input_sr = lava_model.load_audio(str(input_wav))
    print(f"LavaSR loaded {input_wav} at inferred input sr {input_sr}")
    output_audio = lava_model.enhance(input_audio, denoise=denoise, batch=batch)
    output_audio = output_audio.detach().cpu().numpy().squeeze()
    sf.write(str(output_wav), output_audio, 48000)
    return output_wav
        


## 8. Pipeline Definitions

Each pipeline returns a final WAV and optionally a remuxed MP4. The shared runner creates a 48 kHz comparison WAV for every successful branch.
        


In [ ]:
def record_result(row):
    RESULTS.append(row)
    return row


def run_pipeline(name, pipeline_fn):
    out_dir = safe_reset_dir(OUTPUT_DIR / name)
    row = {
        "pipeline": name,
        "status": "failed",
        "final_wav": "",
        "comparison_wav": "",
        "remuxed_mp4": "",
        "error": "",
    }
    print("\n" + "=" * 80)
    print("Running pipeline:", name)
    print("=" * 80)
    try:
        final_wav, remuxed_mp4 = pipeline_fn(out_dir)
        comparison_wav = make_comparison_wav(final_wav, out_dir / f"{name}_comparison_48k.wav")
        row.update({
            "status": "ok",
            "final_wav": str(Path(final_wav)),
            "comparison_wav": str(Path(comparison_wav)),
            "remuxed_mp4": str(Path(remuxed_mp4)) if remuxed_mp4 else "",
        })
        print("Pipeline complete:", name)
    except Exception as exc:
        row["error"] = str(exc)[-4000:]
        print("Pipeline failed:", name)
        print(row["error"])
    return record_result(row)


def require_baseline():
    if BASELINE_WAV is None or not Path(BASELINE_WAV).exists():
        raise RuntimeError("Run the baseline extraction cell before model pipelines.")
    return Path(BASELINE_WAV)


def pipeline_deepfilternet_resemble(out_dir):
    baseline = require_baseline()
    df_wav = run_deepfilternet(baseline, out_dir / "deepfilternet")
    resemble_wav = run_resemble(df_wav, out_dir / "resemble")
    return resemble_wav, None


def pipeline_resemble_loudnorm_remux(out_dir):
    baseline = require_baseline()
    resemble_wav = run_resemble(baseline, out_dir / "resemble")
    normalized_wav = loudnorm_wav(resemble_wav, out_dir / "resemble_loudnorm.wav")
    remuxed_mp4 = remux_video(VIDEO_PATH, normalized_wav, out_dir / "resemble_loudnorm_remux.mp4")
    return normalized_wav, remuxed_mp4


def pipeline_resemble_only(out_dir):
    baseline = require_baseline()
    resemble_wav = run_resemble(baseline, out_dir / "resemble")
    return resemble_wav, None


def pipeline_denoise_lavasr_loudnorm(out_dir):
    baseline = require_baseline()
    denoise_choice = DENOISE_FOR_LAVASR.strip().lower()
    if denoise_choice == "deepfilternet":
        denoised_wav = run_deepfilternet(baseline, out_dir / "deepfilternet")
    elif denoise_choice == "resemble_denoise":
        denoised_wav = run_resemble(baseline, out_dir / "resemble_denoise", denoise_only=True)
    else:
        raise ValueError('DENOISE_FOR_LAVASR must be "deepfilternet" or "resemble_denoise"')
    lava_wav = run_lavasr(denoised_wav, out_dir / "denoise_lavasr.wav")
    normalized_wav = loudnorm_wav(lava_wav, out_dir / "denoise_lavasr_loudnorm.wav")
    return normalized_wav, None


def pipeline_lavasr_only(out_dir):
    baseline = require_baseline()
    lava_wav = run_lavasr(baseline, out_dir / "lavasr.wav")
    return lava_wav, None


def pipeline_lavasr_loudnorm(out_dir):
    baseline = require_baseline()
    lava_wav = run_lavasr(baseline, out_dir / "lavasr.wav")
    normalized_wav = loudnorm_wav(lava_wav, out_dir / "lavasr_loudnorm.wav")
    return normalized_wav, None
        


## 9. Run the Six Experiments

This cell may take a while. If one branch fails, the rest continue.
        


In [ ]:
RESULTS = []

PIPELINES = [
    ("deepfilternet_resemble", pipeline_deepfilternet_resemble),
    ("resemble_loudnorm_remux", pipeline_resemble_loudnorm_remux),
    ("resemble_only", pipeline_resemble_only),
    ("denoise_lavasr_loudnorm", pipeline_denoise_lavasr_loudnorm),
    ("lavasr_only", pipeline_lavasr_only),
    ("lavasr_loudnorm", pipeline_lavasr_loudnorm),
]

for pipeline_name, pipeline_fn in PIPELINES:
    run_pipeline(pipeline_name, pipeline_fn)

results_df = pd.DataFrame(RESULTS)
display(results_df)
        


## 10. Listen and Compare

Use headphones if possible. Pay attention to speech clarity, metallic artifacts, phasey sound, breathing/noise pumping, and whether the voice still sounds natural.
        


In [ ]:
def show_results():
    if not RESULTS:
        print("No results yet. Run the six experiment cell first.")
        return

    results_df = pd.DataFrame(RESULTS)
    display(results_df)

    if Path(BASELINE_WAV).exists():
        display(HTML("<h3>Baseline audio from extracted video</h3>"))
        display(HTML("<b>Original extracted WAV, 48 kHz PCM</b>"))
        display(Audio(filename=str(BASELINE_WAV)))
    else:
        display(HTML("<p>Baseline audio is not available. Run the extraction cell first.</p>"))

    for row in RESULTS:
        title = f"<h3>{row['pipeline']} - {row['status']}</h3>"
        display(HTML(title))
        if row["status"] != "ok":
            display(HTML(f"<pre>{row['error']}</pre>"))
            continue
        display(HTML("<b>Comparison WAV, 48 kHz PCM</b>"))
        display(Audio(filename=row["comparison_wav"]))
        if row.get("remuxed_mp4"):
            display(HTML("<b>Remuxed MP4</b>"))
            display(Video(row["remuxed_mp4"], embed=False))


show_results()
        


## 11. Archive and Download Outputs
        


In [ ]:
archive_path = shutil.make_archive("/content/audio_experiment_outputs", "zip", ROOT)
print("Archive created:", archive_path)

try:
    from google.colab import files
    files.download(archive_path)
except Exception as exc:
    print("Automatic download did not start. Use the Files sidebar to download:")
    print(archive_path)
    print("Download error:", exc)
        


## Optional Rescue Branches Not Run by Default

These models are intentionally excluded from the default notebook run because the first exploration should compare the six requested branches without making the environment heavier than necessary.

- VoiceFixer: useful for clipped, reverberant, or very degraded speech.
- VoiceRestore: experimental restoration for severe degradation, distortion, signal loss, and heavy reverb.
- AudioSR / Versatile Audio Super Resolution: useful for non-speech or mixed audio restoration, but slower and easier to overuse on clean speech.

Add these only after listening to the default outputs and identifying the failure mode that still needs work.
        
